In [ ]:
# pip install openai 
# pip install -e "/path/to/runledger/packages/sdk[openai]"

In [10]:
from __future__ import annotations
import sys
import openai

In [11]:
from runledger_sdk import RunLedger

In [12]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"  # change to whichever model you have pulled
use_local_mode = "--local" in sys.argv
RUNLEDGER_API_KEY="rl_test_xzBOCyeo8m1dhZD7oMNRbisHzeff1OmYJtft2J09Lmk"

In [13]:
# ── 1. RunLedger client ───────────────────────────────────────────────────────
#
# local=True  → prints events as JSON to stdout, no API key needed
# local=False → sends to RunLedger API (set RUNLEDGER_API_KEY env var)
rl = RunLedger(
    api_key=RUNLEDGER_API_KEY,   # reads RUNLEDGER_API_KEY env var automatically
    local=False,     # remove this line once you have a live API running
)


In [14]:
# ── 2. Instrument — patches openai.OpenAI so every call is captured ───────────
rl.instrument()


In [15]:
# ── 3. OpenAI client pointed at Ollama ────────────────────────────────────────
#
# Ollama exposes an OpenAI-compatible REST API at /v1.
# api_key can be any non-empty string — Ollama ignores it.
client = openai.OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
)


In [16]:
# ── Agent: simple multi-turn Q&A ──────────────────────────────────────────────

def run_chat(user_id: str, questions: list[str]) -> None:
    """Send a sequence of questions, maintaining conversation history."""

    history: list[dict[str, str]] = [
        {"role": "system", "content": "You are a concise, helpful assistant."},
    ]

    with rl.context(
        end_user_id=user_id,
        feature_tag="ollama-demo",
        deployment_version="v1.0",
    ) as run_id:
        print(f"\n[RunLedger] run_id={run_id}")
        print(f"[Model]     {MODEL}  ({OLLAMA_BASE_URL})\n")

        for question in questions:
            history.append({"role": "user", "content": question})

            response = client.chat.completions.create(
                model=MODEL,
                messages=history,  # type: ignore[arg-type]
                temperature=0.7,
            )

            answer = response.choices[0].message.content or ""
            history.append({"role": "assistant", "content": answer})

            print(f"Q: {question}")
            print(f"A: {answer}")
            print()

In [17]:
if __name__ == "__main__":
    run_chat(
        user_id="user-local",
        questions=[
            "What is a large language model? One sentence.",
            "Give me one real-world use case for it.",
            "What is the main cost driver when running these at scale?",
        ],
    )

    # Flush all buffered events before exit
    rl.shutdown()


[RunLedger] run_id=9ddd1b06-6c3f-4673-8c54-d73bb085b643
[Model]     llama3.2  (http://localhost:11434/v1)

Q: What is a large language model? One sentence.
A: A large language model (LLM) is a type of artificial intelligence designed to process and understand human language by analyzing vast amounts of text data, generating responses, and learning patterns in natural language processing.

{"event_type": "run_start", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "started_at": "2026-03-01T04:04:39.042621+00:00", "end_user_id": "user-local", "feature_tag": "ollama-demo", "deployment_version": "v1.0"}
{"event_type": "provider_call", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "provider": "openai", "model": "llama3.2", "latency_ms": 1183, "status": "success", "input_tokens": 43, "output_tokens": 41}
{"event_type": "run_end", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "status": "succeeded", "ended_at": "2026-03-01T04:04:40.226740+00:00", "total_input_tokens": 43, "total_out

{"event_type": "provider_call", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "provider": "openai", "model": "llama3.2", "latency_ms": 1226, "status": "success", "input_tokens": 103, "output_tokens": 49}
{"event_type": "run_end", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "status": "succeeded", "ended_at": "2026-03-01T04:04:41.454158+00:00", "total_input_tokens": 103, "total_output_tokens": 49}
{"event_type": "run_start", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "started_at": "2026-03-01T04:04:41.454293+00:00", "end_user_id": "user-local", "feature_tag": "ollama-demo", "deployment_version": "v1.0"}
{"event_type": "provider_call", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "provider": "openai", "model": "llama3.2", "latency_ms": 1021, "status": "success", "input_tokens": 173, "output_tokens": 47}
{"event_type": "run_end", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "status": "succeeded", "ended_at": "2026-03-01T04:04:42.476242+00:00", "total_input_tokens